# Equity volatility calibration
From the listed option quotes of a Bloomberg snapshot to the implied vol surface and the repo curve, step by step on
the real code. Same sections as Models and Pricing, Market handles, *Equity volatilities* (§1.4); AAPL for a single
stock (American options), SX5E for an index (European options, dividends implied by the options). Data: the
2026-11-13 snapshot of the two underlyings, fixed in `equity_vol_calibration_data.csv` (one row per row of a snapshot
table, `table` = its file).

In [ ]:
import os
import sys
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "marketdata"))                   # mdbuild: the download rules

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import yaml
from scipy.stats import norm
import tensorquant as tq
from tensorquant.calibration import equityvol as ev             # internal steps of build_equity_vol (ev._...)
from mdbuild import options as mdo

DATA = pd.read_csv("equity_vol_calibration_data.csv", low_memory=False)
frames = {name: g.drop(columns="table").dropna(axis=1, how="all").reset_index(drop=True)
          for name, g in DATA.groupby("table")}
frames["eq_spot.csv"] = frames["eq_spot.csv"].reindex(columns=["ticker", "ccy", "spot", "repo", "div_yield"])
QUOTES = frames["eq_option_quotes.csv"]
EVALUATION_DATE = pd.Timestamp(QUOTES["quote_date"].iloc[0]).date()
mkt = tq.MarketEnvironment.from_frames(frames, EVALUATION_DATE)  # the snapshot of the two underlyings
tq.Settings.evaluation_date = EVALUATION_DATE
OPTIONS = yaml.safe_load((REPO / "marketdata" / "config" / "settings.yaml").read_text())["options"]
TICKER, INDEX = "AAPL_US", "SX5E"
print("tables:", {k: len(v) for k, v in frames.items()})
QUOTES.groupby("ticker").agg(options=("strike", "size"), expiries=("expiry", "nunique"),
                             exercise=("exercise", "first"), quote_date=("quote_date", "first"),
                             ref_spot=("ref_spot", "first"), spot_source=("spot_source", "first"))

## 1. Construction from listed options
One expiry at a time: the forward of the expiry (put-call parity, §3), then its smile (§5-6), then the surface across
the expiries (§7). The input is one table of quotes per underlying, the output the surface, the repo curve and a report.

In [ ]:
def equity_inputs(ticker):
    """Quotes of *ticker* and its view as in calibrate_snapshot_vols: spot of the quotes, repo left to the options."""
    q = QUOTES[QUOTES["ticker"] == ticker].copy()
    q["expiry"] = pd.to_datetime(q["expiry"]).dt.date
    e = mkt.get_equity(ticker)
    spot = tf.Variable(float(q["ref_spot"].iloc[0]), dtype=tf.float64)
    return q, tq.Equity(ticker, e.currency, spot, e.discount_curve, None, e.dividends, e.div_yield, None)


quotes, view = equity_inputs(TICKER)
eq = mkt.get_equity(TICKER)
QUOTE_DATE = pd.Timestamp(quotes["quote_date"].iloc[0]).date()
REF_SPOT = float(view.spot)
print(f"{TICKER}: {len(quotes)} quotes, spot {REF_SPOT} ({quotes['spot_source'].iloc[0]}), "
      f"{len(eq.dividends.ex_dates) if eq.dividends else 0} projected dividends")
quotes.head(8)

## 2. The quotes
### 2.1 Expiries
The first `weekly_expiries` listed, the monthlies (third Friday) to `monthly_months`, the quarterly monthlies to
`quarterly_months`, then one a year (nearest to mid-December) to `max_years`.

In [ ]:
print({k: OPTIONS[k] for k in ("weekly_expiries", "monthly_months", "quarterly_months", "max_years")})
expiries = sorted(quotes["expiry"].unique())
first = expiries[:OPTIONS["weekly_expiries"]]
monthly_end = mdo._add_months(QUOTE_DATE, OPTIONS["monthly_months"])
quarterly_end = mdo._add_months(QUOTE_DATE, OPTIONS["quarterly_months"])


def rule(e):
    if e in first:
        return "first listed"
    if mdo._monthly(e) and e <= monthly_end:
        return "monthly"
    if mdo._monthly(e) and e.month in (3, 6, 9, 12) and e <= quarterly_end:
        return "quarterly"
    return "yearly (nearest to 15 Dec)"


reselected = mdo.select_expiries(expiries, QUOTE_DATE, OPTIONS["weekly_expiries"], OPTIONS["monthly_months"],
                                 OPTIONS["quarterly_months"], OPTIONS["max_years"])
print("select_expiries keeps every downloaded expiry:", reselected == expiries)
pd.DataFrame({"expiry": expiries, "days": [(e - QUOTE_DATE).days for e in expiries], "rule": [rule(e) for e in expiries]})

### 2.2 Strikes
Around $\hat F = (S_0 - \text{dividends to } T)\,e^{rT}$ (overnight fixing): the nearest out-of-the-money option to each
target moneyness of $\hat F$, plus a call and a put on the `parity_strikes` strikes each side of $\hat F$.

In [ ]:
print("moneyness targets:", OPTIONS["moneyness"], "| parity strikes:", OPTIONS["parity_strikes"])
fixings = frames["ir_fixings.csv"]
rate = float(fixings.loc[fixings["ccy"] == quotes["ccy"].iloc[0], "value"].iloc[0])      # overnight fixing (SOFR)
divs = frames["eq_dividends.csv"].query("ticker == @TICKER")
f_hat = {e: mdo.forward_estimate(REF_SPOT, divs, rate, QUOTE_DATE, e) for e in expiries}
quotes["m_hat"] = quotes["strike"] / quotes["expiry"].map(f_hat)
both = quotes.groupby(["expiry", "strike"])["option_type"].transform("nunique") == 2
quotes["role"] = np.where(both, "parity (C and P)", "smile (OTM)")
summary = quotes.groupby("expiry").agg(calls=("option_type", lambda s: (s == "C").sum()),
                                       puts=("option_type", lambda s: (s == "P").sum()),
                                       parity_strikes=("role", lambda s: (s == "parity (C and P)").sum() // 2),
                                       m_min=("m_hat", "min"), m_max=("m_hat", "max"))
summary.insert(0, "F_hat", summary.index.map(f_hat))
summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = {e: i for i, e in enumerate(expiries)}
for (cp, role), g in quotes.groupby(["option_type", "role"]):
    ax.scatter(g["expiry"].map(x) + (0.12 if cp == "C" else -0.12), g["m_hat"], s=14,
               marker="o" if role.startswith("smile") else "s", label=f"{cp} {role}")
ax.axhline(1, color="grey", lw=0.8)
ax.set_xticks(range(len(expiries)), [str(e) for e in expiries], rotation=60)
ax.set_ylabel("K / F_hat"); ax.set_title(f"{TICKER}: the requested strikes"); ax.legend(fontsize=8); plt.show()

### 2.3 Quote date, spot and stale quotes
Quote date = the most frequent update date of the two-sided quotes; spot read with the options (the close of that day if
it is not today). A bid / ask older than `stale_days` before it was dropped at download: the option stays on its last
price (settlement) or is dropped.

In [ ]:
print("stale_days:", OPTIONS["stale_days"], "| quote date", QUOTE_DATE, "| spot", REF_SPOT, quotes["spot_source"].iloc[0])
quotes["kind"] = np.select([quotes["bid"].notna() & quotes["ask"].notna(), quotes["bid"].notna() | quotes["ask"].notna(),
                            quotes["last"] > 0], ["bid / ask", "one-sided", "last only (settlement)"], "nothing")
cols = ["last_update"] + (["last_update_time"] if "last_update_time" in quotes else [])
display(pd.crosstab(quotes["expiry"], quotes["kind"]))
quotes.groupby("kind")[cols].agg(lambda s: s.value_counts().head(3).to_dict())

### 2.4 Which quotes enter the fit
Two-sided: $0 <$ bid $\le$ ask $\le 3\times$ bid, at the mid, weight 1. Settlement (no bid nor ask, last $> 0$): bid = ask
= last, weight 0.25. Everything else is discarded.

In [ ]:
bid, ask, last = quotes["bid"], quotes["ask"], quotes["last"]
quotes["status"] = np.select(
    [(bid > 0) & (ask >= bid) & (ask <= 3 * bid), bid.isna() & ask.isna() & (last > 0), (bid > 0) & (ask > 3 * bid),
     (bid > 0) & (ask < bid)],
    ["two-sided (w = 1)", "settlement (w = 0.25)", "too wide: ask > 3 bid", "crossed"], "one-sided / zero bid")
prepared = ev._prepare_quotes(quotes)                               # the filter of build_equity_vol
print("kept here:", quotes["status"].str.startswith(("two", "sett")).sum(), "| kept by _prepare_quotes:", len(prepared))
pd.crosstab(quotes["expiry"], quotes["status"], margins=True)

## 3. Forward and repo of an expiry
Parity forward $F = K + (C - P)/P(0,T)$ at the two strikes that bracket $F$, American mids turned European first, iterated
to a fixed point. The repo is the rest of the carry: $q_r(T) = -\frac1T\ln\frac{F\,P(0,T)}{S_0 - \mathrm{PV_{div}}(T)}$.
### 3.1 One expiry by hand

In [ ]:
EXPIRY = min(expiries, key=lambda e: abs((e - QUOTE_DATE).days - 365))   # about one year
grp = prepared[prepared["expiry"] == EXPIRY]
market = grp[~grp["settlement"]]
res = tq.implied_forward(view, EXPIRY, market)                     # dividend_model "continuous", as build_equity_vol
t = view.year_fraction(EXPIRY)
df = float(view.discount_curve.discount(EXPIRY))
r = -np.log(df) / t
pv_div = float(view.pv_dividends(EXPIRY))
F = res["forward"]
print(f"{EXPIRY}: T = {t:.4f}, P(0,T) = {df:.6f}, PV div = {pv_div:.4f}")
{k: res[k] for k in ("forward", "repo", "forward_bid", "forward_ask", "strike_low", "strike_high", "iterations", "converged")}

De-Americanization at the final strike pair: the American mid's implied vol on $F$ (Bjerksund-Stensland, spot $S_0$,
carry $q = r - \ln(F/S_0)/T$), the European price at that vol, the parity forward from the European prices.

In [ ]:
def bs(S, K, t, r, q, vol, cp):
    d1 = (np.log(S / K) + (r - q + 0.5 * vol**2) * t) / (vol * np.sqrt(t))
    d2 = d1 - vol * np.sqrt(t)
    call = S * np.exp(-q * t) * norm.cdf(d1) - K * np.exp(-r * t) * norm.cdf(d2)
    return call if cp == "C" else call - S * np.exp(-q * t) + K * np.exp(-r * t)


q_carry = r - np.log(F / REF_SPOT) / t
pair = np.array([res["strike_low"], res["strike_high"]])
rows = {}
for cp in ("C", "P"):
    side = market[market["option_type"] == cp].set_index("strike").loc[pair]
    mid = 0.5 * (side["bid"] + side["ask"]).to_numpy()
    vol = tq.implied_volatility(mid, pair, cp, REF_SPOT, r, q_carry, t, american=True)
    european = bs(REF_SPOT, pair, t, r, q_carry, vol, cp)
    rows[cp] = european
    print(cp, "mid", mid.round(4), "| American vol", vol.round(4), "| European", european.round(4),
          "| early exercise premium", (mid - european).round(4))
parity = pair + (rows["C"] - rows["P"]) / df
print("parity forwards", parity.round(5), "-> mean", parity.mean().round(5), "| implied_forward", round(F, 5))
print(f"repo = -ln(F P / (S - PV div)) / T = {-np.log(F * df / (REF_SPOT - pv_div)) / t:.5%} (implied_forward {res['repo']:.5%})")

### 3.2 Every expiry: forwards, bid / ask range, repo
Repo pillars only from 30 days (a small forward error is a large rate over a few days).

In [ ]:
fwd_rows = []
for e, g in prepared.groupby("expiry"):
    m = g[~g["settlement"]]
    try:
        o = tq.implied_forward(view, e, m if len(m) else g)
    except ValueError as exc:
        fwd_rows.append({"expiry": e, "error": str(exc)[:60]}); continue
    fwd_rows.append({"expiry": e, "days": (e - QUOTE_DATE).days, "source": "market" if len(m) else "settlement",
                     "F_hat": f_hat[e], "forward": o["forward"],
                     "range bp": 1e4 * (o["forward_ask"] - o["forward_bid"]) / o["forward"],
                     "repo %": 100 * o["repo"], "iter": o["iterations"], "converged": o["converged"]})
forwards = pd.DataFrame(fwd_rows)
forwards.round(4)

In [ ]:
stored_repo = frames["eq_repo.csv"].query("ticker == @TICKER")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ok = forwards["days"] >= 30
ax[0].plot(forwards["days"], forwards["repo %"], "o:", label="every expiry")
ax[0].plot(forwards.loc[ok, "days"], forwards.loc[ok, "repo %"], "o-", label="pillars (>= 30 days)")
ax[0].plot([(pd.Timestamp(p).date() - QUOTE_DATE).days for p in stored_repo["pillar"]], 100 * stored_repo["rate"], "x",
           color="k", label="eq_repo.csv")
ax[0].set_xlabel("days"); ax[0].set_ylabel("repo %"); ax[0].legend(fontsize=8); ax[0].set_title("Repo per expiry")
ax[1].bar(forwards["days"].astype(str), forwards["range bp"]); ax[1].set_title("Bid / ask range of the forward (bp of F)")
ax[1].tick_params(axis="x", rotation=60); plt.tight_layout(); plt.show()

## 4. Index dividends: implied by the options, the repo held
SX5E: projections kept to `DIVIDEND_KEEP_YEARS`; repo $q_0$ held from the last listed expiry inside them; beyond, each
expiry gives $\mathrm{PV_{div}}(T) = S_0 - F(T)\,P(0,T)\,e^{q_0T}$, spread with the weekly pattern; constant yield to 10Y.

In [ ]:
iq = QUOTES[QUOTES["ticker"] == INDEX].copy()
download = frames["eq_dividends_download.csv"].query("ticker == @INDEX").sort_values("ex_date")
ieq = mkt.get_equity(INDEX)
idx_div = tq.DividendCurve.from_dataframe(reference_date=mkt.evaluation_date, df=download, ex_date_col="ex_date",
                                          amount_col="amount", declared_date_col=None, payment_date_col="pay_date",
                                          currency=ieq.currency, daycounter_convention=tq.DayCounterConvention.Actual365)
iview = tq.Equity(INDEX, ieq.currency, tf.Variable(float(iq["ref_spot"].iloc[0]), dtype=tf.float64),
                  ieq.discount_curve, None, idx_div, None, None)
used = ev.implied_index_dividends(iview, iq)
print("keep years:", ev.DIVIDEND_KEEP_YEARS, "| cutoff", used.attrs["cutoff"], f"| repo held {used.attrs['repo']:.4%}",
      f"| extrapolated yield {used.attrs.get('yield', float('nan')):.4%}")
stored = frames["eq_dividends.csv"].query("ticker == @INDEX")
print("same as eq_dividends.csv:", len(stored) == len(used) and np.allclose(stored["amount"], used["amount"]))
pd.DataFrame(used.attrs["implied"]).round(3)

In [ ]:
def by_year(frame, label):
    return frame.assign(year=pd.to_datetime(frame["ex_date"]).dt.year).groupby("year")["amount"].sum().rename(label)


years = pd.concat([by_year(download, "download")] +
                  [by_year(used[used["source"] == s], s) for s in ("download", "implied", "extrapolated")], axis=1)
years.columns = ["Bloomberg projection", "kept", "implied", "extrapolated"]
years.fillna(0).plot.bar(figsize=(10, 3.5), title=f"{INDEX} dividends per year (index points)", width=0.8)
plt.show()

In [ ]:
used_curve = tq.DividendCurve(mkt.evaluation_date, list(used["ex_date"]), [float(a) for a in used["amount"]],
                              ieq.currency, payment_dates=list(used["pay_date"]),
                              daycounter_convention=tq.DayCounterConvention.Actual365)
ip = ev._prepare_quotes(iq)
fig, ax = plt.subplots(figsize=(8, 3.5))
for label, curve in (("Bloomberg projections", idx_div), ("implied dividends, repo held", used_curve)):
    v = tq.Equity(INDEX, ieq.currency, iview.spot, ieq.discount_curve, None, curve, None, None)
    pts = []
    for e, g in ip.groupby("expiry"):
        m = g[~g["settlement"]]
        if (e - QUOTE_DATE).days >= 30:
            try:
                pts.append(((e - QUOTE_DATE).days / 365, 100 * tq.implied_forward(v, e, m if len(m) else g)["repo"]))
            except ValueError:
                pass
    ax.plot(*zip(*pts), "o-", label=label)
ax.set_xlabel("years"); ax.set_ylabel("implied repo %"); ax.set_title(f"{INDEX}: repo left to the options")
ax.legend(); plt.show()

## 5. The smile model: a mixture of lognormals with a default mass
$S_T = 0$ with probability $Q$, $S_T = \xi_l F e^{\Sigma_l Z - \Sigma_l^2/2}$ with probability $p_l$ ($l = 1..4$);
$Q + \sum p_l = 1$, $\sum p_l\xi_l = 1$. Prices = sums of Black prices (plus $QK$ for a put).

In [ ]:
calib = frames["eq_vol_calibration.csv"].query("ticker == @TICKER")
row = calib[calib["expiry"] == str(EXPIRY)].iloc[0]
mix = tq.LognormalMixture(*(np.array(eval(row[c])) for c in ("weights", "means", "stdevs")), row["default"])
print(pd.DataFrame({"p": mix.weights, "xi": mix.means, "Sigma": mix.stdevs, "vol = Sigma/sqrt(T)": mix.stdevs / np.sqrt(t)}).round(4))
print(f"Q = {mix.default:.4%} | Q + sum p = {mix.default + mix.weights.sum():.12f} | sum p xi = {(mix.weights * mix.means).sum():.12f}")

K = F * np.linspace(0.5, 1.6, 9)
black = lambda f, k, s: f * norm.cdf(np.log(f / k) / s + s / 2) - k * norm.cdf(np.log(f / k) / s - s / 2)
by_hand = sum(p * black(x * F, K, s) for p, x, s in zip(mix.weights, mix.means, mix.stdevs))
print("calls = sum of Black prices:", np.allclose(mix.price(F, K, 1.0, "C"), by_hand))
one = tq.LognormalMixture([1.0], [1.0], [0.3 * np.sqrt(t)], 0.0)
print("one lognormal, no default -> flat Black smile:", one.implied_vols(F, K, t).round(6))

In [ ]:
s = np.linspace(0.01, 2.0, 600) * F
dens = sum(p * norm.pdf(np.log(s / (x * F)) + sg**2 / 2, scale=sg) / s for p, x, sg in zip(mix.weights, mix.means, mix.stdevs))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for p, x, sg in zip(mix.weights, mix.means, mix.stdevs):
    ax[0].plot(s / F, p * norm.pdf(np.log(s / (x * F)) + sg**2 / 2, scale=sg) / s * F, lw=0.8, ls="--")
ax[0].plot(s / F, dens * F, "k", label=f"density (+ mass {mix.default:.2%} at 0)")
ax[0].set_xlabel("S_T / F"); ax[0].legend(fontsize=8); ax[0].set_title(f"{TICKER} {EXPIRY}: risk-neutral density")
k = F * np.linspace(0.4, 1.8, 200)
ax[1].plot(k / F, mix.implied_vols(F, k, t), "k")
ax[1].set_xlabel("K / F"); ax[1].set_ylabel("Black vol"); ax[1].set_title("its smile"); plt.tight_layout(); plt.show()

## 6. Fit of the smile
Out-of-the-money quotes on the implied forward, each turned into a Black vol (American: the vol of its price on that
forward); vols outside $0.2$–$5\times$ the median dropped; fewer than 3 left: expiry skipped. Least squares on
$w_i\,(V_{\text{model}}(K_i) - \text{Black}(F, K_i, \sigma_i))/\mathcal V_i$, i.e. about the vol errors.
### 6.1 One expiry by hand

In [ ]:
otm = grp[np.where(grp["option_type"] == "P", grp["strike"] < F, grp["strike"] >= F)].sort_values("strike")
vols = {}
for name, price in (("bid", otm["bid"]), ("mid", 0.5 * (otm["bid"] + otm["ask"])), ("ask", otm["ask"])):
    vols[name] = np.concatenate([tq.implied_volatility(price[otm["option_type"] == cp].to_numpy(), otm.loc[otm["option_type"] == cp, "strike"].to_numpy(),
                                                       cp, REF_SPOT, r, q_carry, t, american=True) for cp in ("P", "C")])
strikes = np.concatenate([otm.loc[otm["option_type"] == cp, "strike"].to_numpy() for cp in ("P", "C")])
settle = np.concatenate([otm.loc[otm["option_type"] == cp, "settlement"].to_numpy() for cp in ("P", "C")])
v = vols["mid"]
median = np.nanmedian(v)
keep = np.isfinite(v) & (v > 0.2 * median) & (v < 5 * median)
w = np.where(settle, 0.25, 1.0)
fit = tq.LognormalMixture.fit(F, t, strikes[keep], v[keep], n=4, weights=w[keep])
print(f"{keep.sum()} of {len(v)} quotes used (median vol {median:.2%}), {settle.sum()} settlements | rms {fit.rms:.4%}"
      f" (stored {row['fit_rms']:.4%})")
table = pd.DataFrame({"K/F": strikes / F, "settlement": settle, "used": keep, "weight": w,
                      **{f"iv_{n}": vols[n] for n in vols}, "model": fit.implied_vols(F, strikes, t)})
table.assign(**{"error bp": 1e4 * (table["model"] - table["iv_mid"])}).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
kk = F * np.linspace(strikes.min() / F * 0.95, strikes.max() / F * 1.05, 200)
ax.vlines(strikes / F, vols["bid"], vols["ask"], color="grey", lw=2, alpha=0.5, label="bid / ask")
ax.scatter((strikes / F)[keep & ~settle], v[keep & ~settle], s=18, label="mid (w = 1)")
ax.scatter((strikes / F)[keep & settle], v[keep & settle], s=22, marker="s", label="settlement (w = 0.25)")
ax.scatter((strikes / F)[~keep], np.nan_to_num(v[~keep]), marker="x", color="r", label="dropped")
ax.plot(kk / F, fit.implied_vols(F, kk, t), "k", label=f"mixture, rms {fit.rms:.2%}")
ax.set_xlabel("K / F"); ax.set_ylabel("vol"); ax.set_title(f"{TICKER} {EXPIRY}"); ax.legend(fontsize=8); plt.show()

### 6.2 Every expiry (`build_equity_vol`)
The standard start is ordered; above 0.2 vol points rms a warm start from the previous expiry's parameters, the better
fit kept.

In [ ]:
surface, repo_curve, report = tq.build_equity_vol(view, quotes)
good = report[report["mixture"].notna()]
report[["expiry", "t", "forward", "repo", "quotes", "source", "fit_rms", "atm_vol"]].assign(
    default=[m.default if isinstance(m, tq.LognormalMixture) else np.nan for m in report["mixture"]]).round(4)

In [ ]:
n = len(good)
fig, axes = plt.subplots(int(np.ceil(n / 4)), 4, figsize=(14, 2.6 * np.ceil(n / 4)), sharey=False)
for ax, (_, g) in zip(axes.ravel(), good.iterrows()):
    qv = g["quote_vols"]
    x = qv["strike"] / g["forward"]
    ax.vlines(x, qv["iv_bid"], qv["iv_ask"], color="grey", alpha=0.5)
    ax.scatter(x[qv["used"]], qv.loc[qv["used"], "iv_mid"], s=8)
    kk = g["forward"] * np.linspace(x.min(), x.max(), 100)
    ax.plot(kk / g["forward"], g["mixture"].implied_vols(g["forward"], kk, g["t"]), "k", lw=1)
    ax.set_title(f"{g['expiry']}  rms {g['fit_rms']:.2%}", fontsize=9)
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7. The surface
Grid: the listed expiries then one point a year to 6Y × 50–150% of the spot, at forward moneyness $K/F(t)$. Between
expiries PCHIP in total variance $\sigma^2t$; before the first the first smile; after the last the last mixture
carried, raised to $\sigma_H^2(t) = \theta + (v_0 - \theta)\frac{1 - e^{-\kappa t}}{\kappa t}$ when below it; NaN wings
linear in log-moneyness. AAPL is listed to about 2Y: its long end is the carried mixture, raised to $\sigma_H$ where
below it; SX5E is listed to 6Y.

In [ ]:
stored_vol = frames["eq_vol.csv"].query("ticker == @TICKER")
matrix = surface.volatility_matrix.numpy()
print("grid:", len(surface.maturity), "tenors x", len(surface.strike), "strikes | moneyness", np.round(ev.GRID_MONEYNESS, 2))
print("max |surface - eq_vol.csv| =", np.abs(np.sort(matrix.ravel()) - np.sort(stored_vol["vol"].to_numpy())).max())
pd.DataFrame(matrix, index=np.round(surface.maturity, 3), columns=np.round(np.array(surface.strike) / REF_SPOT, 2)).round(4)

In [ ]:
def atm_term(ticker, good, surface, view, ax):
    times, atm = good["t"].to_numpy(float), good["atm_vol"].to_numpy(float)
    sigma_h = ev._heston_atm(times, atm)                              # (kappa, v0, theta) fitted to the ATM vols
    tt = np.linspace(times[0], 6.2, 300)
    after = tt[tt > times[-1]]
    last = good["mixture"].iloc[-1]
    carried = [float(ev._extrapolate(last, times[-1], x).implied_vols(1.0, [1.0], x)[0]) for x in after]
    grid_t = np.array(surface.maturity)
    atm_grid = [float(surface.volatility(strike=float(view.forward(x)), tenor=x)) for x in grid_t]
    ax.plot(times, atm, "o", label="ATM vol of each calibrated expiry")
    ax.plot(tt, [sigma_h(x) for x in tt], label="Heston-like sigma_H(t)")
    if len(after):
        ax.plot(after, carried, "--", label="last mixture carried")
        ax.axvline(times[-1], color="grey", lw=0.6)
    ax.plot(grid_t, atm_grid, "k.", label="surface at the forward")
    ax.set_xlabel("years"); ax.legend(fontsize=8); ax.set_title(f"{ticker}: ATM term structure")


idx_quotes, idx_view = equity_inputs(INDEX)                      # dividends of the market: the implied ones (§4)
idx_surface, _, idx_report = tq.build_equity_vol(idx_view, idx_quotes)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
atm_term(TICKER, good, surface, view, ax[0])
atm_term(INDEX, idx_report[idx_report["mixture"].notna()], idx_surface, idx_view, ax[1])
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
grid_t = np.array(surface.maturity)
for i in range(0, len(grid_t), max(1, len(grid_t) // 8)):
    ax.plot(ev.GRID_MONEYNESS, matrix[i], label=f"{grid_t[i]:.2f}y")
ax.set_xlabel("K / S0"); ax.set_ylabel("vol"); ax.set_title(f"{TICKER}: smiles on the grid"); ax.legend(fontsize=8, ncol=2)
plt.show()

## Exercises
1. Set the settlement weight to 1 and to 0 (`build_equity_vol(..., settlement_weight=...)`): which expiries move, and by
   how much at the wings?
2. Recompute the AAPL repo with `dividend_model="discrete"`: does the forward change? Does the American vol?
3. Take SX5E with the Bloomberg projections only (no implied dividends): how does the long-dated repo look, and what
   would it do to a 5Y forward-start price?